# 確定したcheckpointのtest比較

01のfinal_run_ids.jsonを確定した後に実行する。各checkpointを明示し、hash付きcacheを読む。陽性はNo Finding。ランダム化EOのrateとスコアensembleのAUCは別の結果として報告する。

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
import torch
from omegaconf import OmegaConf

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import local_data_path  # noqa: E402
from analysis.common.predictions import load_cache, stale_reason, write_cache  # noqa: E402
from analysis.minimax_auc_vs_eo.evaluation import (  # noqa: E402
    age_metrics,
    bootstrap_seed_differences,
    randomized_positive_probability,
)

STUDY = ROOT / "analysis/minimax_auc_vs_eo"
RESULTS = STUDY / "results"
RESULTS.mkdir(exist_ok=True)
RUNS = [
    path
    for path in sorted((STUDY / "runs").glob("*"))
    if (path / "run.json").is_file() and json.loads((path / "run.json").read_text()).get("status") == "succeeded"
]

## 固定した選択と予測

親checkpointもstudyへ取り込む。患者IDはsplitの列を使い、欠ける場合はCheXpertのpatient番号をimageパスから抽出する。

In [ ]:
def imported_checkpoint(run, row):
    if row["epoch"] > 0:
        return run, Path(row["checkpoint"])
    parent_id = json.loads((run / "run.json").read_text())["parent_run"]["run_id"]
    parent_run = STUDY / "runs" / parent_id
    if not (parent_run / "run.json").is_file():
        raise ValueError(f"親runをstudyへ取り込む必要がある: {parent_id}")
    return parent_run, Path("checkpoints") / Path(row["checkpoint"]).name


def predictions(run, row, split):
    source, checkpoint = imported_checkpoint(run, row)
    matches = list((STUDY / "cache").glob(f"{source.name}_*_{split}.npz"))
    source_hashes = {}
    path = next(
        (
            path
            for path in matches
            if stale_reason(path, source, checkpoint=checkpoint, source_hashes=source_hashes) is None
        ),
        None,
    )
    if path is None:
        path = write_cache(
            source,
            split,
            STUDY / "cache",
            torch.device("cuda" if torch.cuda.is_available() else "cpu"),
            checkpoint=checkpoint,
            source_hashes=source_hashes,
        )[0]
    config = OmegaConf.load(source / "config.yaml")
    frame = pd.read_csv(local_data_path(config.data.cv_splits_dir) / f"{split}.csv")
    if frame["age_missing"].astype(bool).any():
        raise ValueError("年齢欠損がある")
    cache = load_cache(path, frame.image.to_numpy(dtype=str))
    return (
        cache["logits"][:, 1] - cache["logits"][:, 0],
        cache["target"],
        (frame.age.to_numpy() >= 65).astype(int),
        frame,
    )

In [ ]:
selection = json.loads((RESULTS / "selection.json").read_text()) if (RESULTS / "selection.json").exists() else {}
final_ids = (
    json.loads((RESULTS / "final_run_ids.json").read_text()) if (RESULTS / "final_run_ids.json").exists() else []
)
rows, intervals, registry = [], [], {}
shared_labels = None
for run_id in final_ids:
    run = STUDY / "runs" / run_id
    config = OmegaConf.load(run / "config.yaml")
    method, experiment_seed = config.training_strategy.name, int(config.seed)
    parent_score, y, g, frame = predictions(run, selection[run_id]["parent"], "test")
    patients = (
        frame["patient_id"].to_numpy()
        if "patient_id" in frame
        else frame.image.str.extract(r"(patient\d+)", expand=False).to_numpy()
    )
    assert pd.notna(patients).all()
    if shared_labels is None:
        shared_labels = dict(target=y, groups=g, patient_ids=patients, images=frame.image.to_numpy())
    assert np.array_equal(frame.image, shared_labels["images"]) and np.array_equal(y, shared_labels["target"])
    chosen = selection[run_id]["single"]["selected"]
    if chosen is None:
        rows.append(dict(run_id=run_id, method=method, seed=experiment_seed, status="infeasible"))
    else:
        score, selected_y, selected_g, selected_frame = predictions(run, chosen, "test")
        assert np.array_equal(frame.image, selected_frame.image) and np.array_equal(y, selected_y)
        rows.append(
            dict(run_id=run_id, method=method, seed=experiment_seed, status="single", **age_metrics(score, y, g))
        )
        if experiment_seed in registry.get(method, {}):
            raise ValueError("同じ手法・seedに複数の最終runが指定されている")
        registry.setdefault(method, {})[experiment_seed] = dict(scores=score)
    mixture = selection[run_id].get("eo_mixture", {})
    if mixture.get("status") == "selected":
        components = mixture["components"]
        scores = np.stack([predictions(run, component, "test")[0] for component in components])
        weights = np.array([component["probability"] for component in components])
        probability = randomized_positive_probability(scores, weights)
        rates = age_metrics(parent_score, y, g, positive_probability=probability)
        rows.append(
            dict(
                run_id=run_id,
                method=method,
                seed=experiment_seed,
                status="randomized_eo",
                tpr=rates["tpr"],
                tnr=rates["tnr"],
                tpr_gap=rates["tpr_gap"],
                fpr_gap=rates["fpr_gap"],
            )
        )
        registry.setdefault("randomized_eo", {})[experiment_seed] = dict(
            scores=parent_score, positive_probability=probability
        )
        ensemble = weights @ (1 / (1 + np.exp(-np.clip(scores, -700, 700))))
        rows.append(
            dict(
                run_id=run_id,
                method=method,
                seed=experiment_seed,
                status="probability_score_ensemble",
                **age_metrics(ensemble - 0.5, y, g),
            )
        )
comparisons = [
    ("minimax_auc", "population_auc"),
    ("minimax_auc", "sampling_corrected_erm"),
    ("minimax_auc", "sampling_corrected_group_dro"),
    ("equalized_odds_rate", "sampling_corrected_erm"),
    ("randomized_eo", "sampling_corrected_erm"),
]
for candidate, reference in comparisons:
    seeds = sorted(set(registry.get(candidate, {})) & set(registry.get(reference, {})))
    if seeds != [42, 43, 44]:
        continue
    conditions = []
    for name in (candidate, reference):
        condition = dict(seeds=seeds, scores=np.stack([registry[name][seed]["scores"] for seed in seeds]))
        if name == "randomized_eo":
            condition["positive_probability"] = np.stack(
                [registry[name][seed]["positive_probability"] for seed in seeds]
            )
        conditions.append(condition)
    metrics = (
        ("tpr_gap", "fpr_gap")
        if candidate == "randomized_eo"
        else ("auroc", "worst_age_auroc", "min_pair_auroc", "tpr_gap", "fpr_gap")
    )
    comparison_intervals = bootstrap_seed_differences(*conditions, shared_labels, metrics)
    for metric, result in comparison_intervals.items():
        intervals.append(dict(candidate=candidate, reference=reference, metric=metric, seeds=seeds, **result))
if rows:
    pd.DataFrame(rows).to_csv(RESULTS / "test_comparison.csv", index=False)
if intervals:
    (RESULTS / "paired_bootstrap.json").write_text(json.dumps(intervals, indent=2, allow_nan=False))
print(f"固定済みrun: {len(final_ids)}")

## 比較の解釈

主比較はC7対C6、C8対C3。同seedの差を平均し、3seedのlogitを平均した結果と混同しない。親との差の区間は補助報告。最終的な手法間・3seed平均差も同じ患者抽出を共有して集計する。

## seed別の4ペアAUC

各seedの単一checkpointを線で示す。4種類の陽性群・陰性群ペアを分け、群内と群間の改善を確認する。推論をやり直さず、上の集計結果から描く。

In [ ]:
table = pd.DataFrame(rows)
if not table.empty:
    single = table.loc[table.status == "single"].copy()
    if not single.empty:
        methods = list(single.method.unique())
        positions = np.arange(len(methods))
        figure, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
        pair_names = [
            "positive <65 / negative <65",
            "positive <65 / negative ≥65",
            "positive ≥65 / negative <65",
            "positive ≥65 / negative ≥65",
        ]
        for index, axis in enumerate(axes.flat):
            for experiment_seed, seed_rows in single.groupby("seed", sort=True):
                values = seed_rows.set_index("method").reindex(methods)["pair_auroc"]
                axis.plot(
                    positions,
                    [value[index] if isinstance(value, list) else np.nan for value in values],
                    marker="o",
                    label=f"seed {experiment_seed}",
                )
            axis.set_title(pair_names[index])
            axis.set_ylabel("AUROC")
            axis.set_xticks(positions, methods, rotation=45, ha="right")
            axis.grid(alpha=0.2)
        figure.suptitle("Pair AUROC (positive = No Finding)")
        axes[0, 0].legend()
        figure.tight_layout()
        (STUDY / "figures").mkdir(exist_ok=True)
        figure.savefig(STUDY / "figures/pair_auc_by_seed.png", dpi=200)
        plt.show()

## seed別のTPR・FPR gap

共通閾値0の単一checkpointと、validationで固定したランダム化EOの期待rateを区別する。点線は主実験の許容差0.10。

In [ ]:
if not table.empty:
    rate_rows = table.loc[table.status.isin(["single", "randomized_eo"])].copy()
    if not rate_rows.empty:
        rate_rows["condition"] = rate_rows.method + np.where(rate_rows.status == "randomized_eo", " / mixture", "")
        conditions = list(rate_rows.condition.unique())
        positions = np.arange(len(conditions))
        figure, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
        for axis, metric in zip(axes, ("tpr_gap", "fpr_gap"), strict=True):
            for experiment_seed, seed_rows in rate_rows.groupby("seed", sort=True):
                values = seed_rows.set_index("condition").reindex(conditions)[metric]
                axis.plot(positions, values, marker="o", label=f"seed {experiment_seed}")
            axis.axhline(0.1, color="gray", linestyle="--", label="epsilon=0.10")
            axis.set_title(metric)
            axis.set_xticks(positions, conditions, rotation=45, ha="right")
            axis.grid(alpha=0.2)
        axes[0].set_ylabel("absolute rate gap")
        axes[0].legend()
        figure.tight_layout()
        (STUDY / "figures").mkdir(exist_ok=True)
        figure.savefig(STUDY / "figures/rate_gap_by_seed.png", dpi=200)
        plt.show()